In [ ]:
import pandas as pd
import numpy as np
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from tqdm import tqdm

In [2]:
columns = ['target', 'ids', 'date', 'flag', 'user', 'text']
df = pd.read_csv("Twitter_data.csv", encoding='latin1', names=columns)

In [3]:
df.shape

(1600000, 6)

In [4]:
df.head()

,target,ids,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [5]:
df = df.drop(['ids', 'date', 'flag', 'user'], axis=1)

In [6]:
df.head()

,target,text
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,is upset that he can't update his Facebook by ...
2,0,@Kenichan I dived many times for the ball. Man...
3,0,my whole body feels itchy and like its on fire
4,0,"@nationwideclass no, it's not behaving at all...."


In [7]:
df["target"].value_counts()

target
0    800000
4    800000
Name: count, dtype: int64

In [8]:
df["target"] = df["target"].replace(4, 1)

In [9]:
df["target"].value_counts()

target
0    800000
1    800000
Name: count, dtype: int64

In [11]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /home/sadik/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [14]:
len(stopwords.words('english'))

198

In [15]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

In [16]:
len(stop_words)

198

In [17]:
important_words = {"not", "no", "nor", "don't", "didn't", "won't", "can't", "shouldn't", "wouldn't", "couldn't"}
stop_words = stop_words - important_words

In [18]:
len(stop_words)

189

In [19]:
def preprocess_text(text):
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"[^a-zA-Z]", " ", text)
    words = text.split()
    words = [stemmer.stem(word) for word in words if word.lower() not in stop_words]
    return ' '.join(words)

In [25]:
# apply tqdm for seeing the progress
tqdm.pandas()
df["cleaned_text"] = df["text"].progress_apply(preprocess_text)

100%|██████████| 1600000/1600000 [01:26<00:00, 18405.61it/s]


In [26]:
df.head()

,target,text,cleaned_text
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",awww that bummer shoulda got david carr third day
1,0,is upset that he can't update his Facebook by ...,upset cant updat facebook text might cri resul...
2,0,@Kenichan I dived many times for the ball. Man...,dive mani time ball manag save rest go bound
3,0,my whole body feels itchy and like its on fire,whole bodi feel itchi like fire
4,0,"@nationwideclass no, it's not behaving at all....",no not behav im mad cant see


In [27]:
df = df[df["cleaned_text"] != ""]

In [28]:
df.head()

,target,text,cleaned_text
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",awww that bummer shoulda got david carr third day
1,0,is upset that he can't update his Facebook by ...,upset cant updat facebook text might cri resul...
2,0,@Kenichan I dived many times for the ball. Man...,dive mani time ball manag save rest go bound
3,0,my whole body feels itchy and like its on fire,whole bodi feel itchi like fire
4,0,"@nationwideclass no, it's not behaving at all....",no not behav im mad cant see


In [30]:
df.to_csv("cleaned_twitter_data.csv", index=False)